In [ ]:
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from utils import (
    FloatQuantLinearModel,
    IntQuantLinearModel,
    get_mnist_dataloaders,
    test_model,
    train_for_epoch,
)

DATASET_PATH = "./data"  # Change me
EXPORT_PATH = "./model_outputs"  # Change me

In [ ]:
# Training parameters
learning_rate = 0.01
momentum = 0.5

# ++++ Quantization method (comment/uncomment) ++++
# Int quant
model = IntQuantLinearModel(input_bit_width=4, weight_bit_width=4)

# Float quant
model = FloatQuantLinearModel(
    input_exponent_bit_width=2,
    input_mantissa_bit_width=1,
    weight_exponent_bit_width=4,
    weight_mantissa_bit_width=3,
)

# ---- Quantization method ----

device = (
    "xpu"
    if torch.xpu.is_available()  # For the only person who has an Intel GPU
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, momentum=momentum)

In [ ]:
train_loader, test_loader = get_mnist_dataloaders(
    DATASET_PATH, pin_memory=True, pin_memory_device=device
)

In [ ]:
for epoch in range(10):
    print(f"Epoch: {epoch}")
    train_for_epoch(model, device, train_loader, criterion, optimizer)
    test_model(model, device, test_loader, F.cross_entropy)

In [ ]:
# Get all correctly classified images/inputs from test dataset
correct_inputs, correct_targets = test_model(
    model, device, test_loader, F.cross_entropy, return_correct=True
)

# Get quantized weights and inputs
model_weights, model_scale = model.quant_weight()
correct_inputs, input_scale = model.quant_input(correct_inputs)

In [ ]:
# Create output directory if it doesn't exist
if not os.path.exists(EXPORT_PATH):
    os.makedirs(EXPORT_PATH)

# Export NumPy for compatibility with Rust
np.save(EXPORT_PATH + "/model_weights.npy", model_weights.cpu().numpy())
np.save(EXPORT_PATH + "/model_scale.npy", model_scale.cpu().numpy())
np.save(EXPORT_PATH + "/correct_inputs.npy", correct_inputs.cpu().numpy())
np.save(EXPORT_PATH + "/input_scale.npy", input_scale.cpu().numpy())
np.save(EXPORT_PATH + "/correct_targets.npy", correct_targets.cpu().numpy())